# Load and inspect data

In [48]:
import pandas as pd

#url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins.csv"

#pd.read_csv(url).to_csv("data/penguins.csv", index=False)

df = pd.read_csv('data/penguins.csv')

In [49]:
df = pd.read_csv('data/penguins.csv')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), str(3)
memory usage: 27.6 KB


In [51]:
df.describe(include='all')

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
count,344,344,342.000000,342.000000,342.000000,342.000000,333,344.000000
unique,3,3,NaN,NaN,NaN,NaN,2,NaN
top,Adelie,Biscoe,NaN,NaN,NaN,NaN,male,NaN
freq,152,168,NaN,NaN,NaN,NaN,168,NaN
mean,NaN,NaN,43.921930,17.151170,200.915205,4201.754386,NaN,2008.029070
std,NaN,NaN,5.459584,1.974793,14.061714,801.954536,NaN,0.818356
min,NaN,NaN,32.100000,13.100000,172.000000,2700.000000,NaN,2007.000000
25%,NaN,NaN,39.225000,15.600000,190.000000,3550.000000,NaN,2007.000000
50%,NaN,NaN,44.450000,17.300000,197.000000,4050.000000,NaN,2008.000000
75%,NaN,NaN,48.500000,18.700000,213.000000,4750.000000,NaN,2009.000000


In [52]:
t = '---------------------------------------'

# Audit missing data
Report how many missing values exist in each column, and what percentage of rows are affected overall.

In [53]:
row_missing_count = df.isna().sum()
row_missing_pct = round(df.isna().any(axis=1).mean() * 100,2)

print(row_missing_count)
print(t)
print(row_missing_pct)

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64
---------------------------------------
3.2


# Fill numeric gaps
Fill missing bill_length_mm, bill_depth_mm, flipper_length_mm, and body_mass_g using the median value for that penguin's species (not the overall median).

In [54]:
numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
df.groupby('species')[numeric_cols].median()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
species,,,,
Adelie,38.80,18.40,190.0,3700.0
Chinstrap,49.55,18.45,196.0,3700.0
Gentoo,47.30,15.00,216.0,5000.0


In [55]:
(df.loc[df['bill_length_mm'].isna()])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
271,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,2009


In [56]:
(df.loc[df['bill_depth_mm'].isna()])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
271,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,2009


In [57]:
(df.loc[df['flipper_length_mm'].isna()])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
271,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,2009


In [58]:
(df.loc[df['body_mass_g'].isna()])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
271,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,2009


In [59]:
for col in numeric_cols:
    group_median = df.groupby('species')[col].transform('median')
    df.fillna({col:group_median}, inplace=True)
print(df[numeric_cols].isna().sum())

bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
dtype: int64


Why `transform` here instead of just `groupby().mean()`? Because `transform` returns a Series aligned back to the original index/shape, so you can feed it straight into `fillna()` row-by-row — each missing value gets its own group's mean instead of a single overall number.

Try it on one column first, check with `.isna().sum()` before and after to confirm it worked, then think about whether you want to loop this over all the numeric measurement columns.

# Handle missing sex
Rows with a missing sex value can't be reliably guessed — drop only those rows, and report how many were removed.

In [60]:
missing_sex_count = df.loc[df.sex.isna()].sex.size
df.dropna(subset='sex',inplace=True)
missing_sex_count

11

# Fix inconsistent labels
Standardize the sex column so values are consistently 'male'/'female' lowercase, and fix any stray typos or casing issues in species or island.

PS: This step isn't nessary because all the data in the `sex` column is already lowercase

In [61]:
df['sex'] = df['sex'].str.lower()
df['sex']

0        male
1      female
2      female
4      female
5        male
        ...  
339      male
340    female
341      male
342      male
343    female
Name: sex, Length: 333, dtype: str

# Flag outliers
Add a boolean column `is_outlier` that flags rows where `body_mass_g` is more than `3 standard deviations` from its species' `mean`.

PS: Think about what the actual `standard deviations from the mean` rule needs:

something to subtract `mean`

something to measure distance regardless of direction `absolute value`

then compare that distance to `3 * std`

In [62]:
#df['is_outlier'] = (df['body_mass_g'] - df['body_mass_g'].mean()).abs() > df['body_mass_g'].std() * 3
#df.is_outlier.value_counts()
# This calculation uses one mean and standard deviation for the entire dataset.
# The task requires each penguin to be compared with the mean and standard deviation of its own species.

In [63]:
df['is_outlier'] = False
species_to_check = df.species.unique()
for specie in species_to_check:
    specie_body_mass = df.loc[df['species'] == specie, 'body_mass_g']
    outlier_result = ((specie_body_mass - specie_body_mass.mean()).abs()) > (specie_body_mass.std() * 3)
    df.loc[df['species'] == specie, 'is_outlier'] = outlier_result
df.is_outlier.unique()

array([False])

# Fix data types
Convert year to a proper integer type and species/island/sex to category dtype to save memory.

In [64]:
df['year'] = df['year'].astype('int64')
df[['island', 'sex', 'species']] = df[['island', 'sex', 'species']].astype('category')
df.dtypes

species              category
island               category
bill_length_mm        float64
bill_depth_mm         float64
flipper_length_mm     float64
body_mass_g           float64
sex                  category
year                    int64
is_outlier               bool
dtype: object

# Create a size index
Add a column body_mass_kg converting grams to kilograms, and a bill_ratio column = bill_length_mm / bill_depth_mm.

In [65]:
df['body_mass_kg'] = df.body_mass_g / 1000
df['bill_ratio'] = df.bill_length_mm / df.bill_depth_mm
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,is_outlier,body_mass_kg,bill_ratio
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007,False,3.75,2.090909
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007,False,3.80,2.270115
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007,False,3.25,2.238889
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007,False,3.45,1.901554
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007,False,3.65,1.907767


# Bucket into size classes
Add a size_class column: 'small', 'medium', or 'large' based on body_mass_g terciles (roughly equal-sized groups).

In [66]:
df['size_class'] = pd.qcut(df.body_mass_g,3,labels=['small','medium','large'])
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,is_outlier,body_mass_kg,bill_ratio,size_class
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007,False,3.75,2.090909,medium
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007,False,3.80,2.270115,medium
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007,False,3.25,2.238889,small
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007,False,3.45,1.901554,small
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007,False,3.65,1.907767,small


# Rename for the report
Rename columns to report-friendly labels, e.g. bill_length_mm → Bill Length (mm).

In [67]:
df.rename(columns={ 'bill_length_mm':'Bill Length (mm)',
                    'bill_depth_mm':'Bill Depth (mm)',
                    'flipper_length_mm':'Flipper Length (mm)',
                    'body_mass_g':'Body Mass (g)',
                    'species':'Species'},
                    inplace=True)
df['sex'] = df.sex.str.title()
df.head()

,Species,island,Bill Length (mm),Bill Depth (mm),Flipper Length (mm),Body Mass (g),sex,year,is_outlier,body_mass_kg,bill_ratio,size_class
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,2007,False,3.75,2.090909,medium
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,2007,False,3.80,2.270115,medium
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,2007,False,3.25,2.238889,small
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,2007,False,3.45,1.901554,small
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male,2007,False,3.65,1.907767,small


# Species summary table
Build a summary table with one row per species showing count, mean body mass, mean flipper length, and mean bill length.x

In [68]:
spec_summ = df.groupby('Species').agg({ 'Species':'count',
                                        'Body Mass (g)':'mean',
                                        'Bill Length (mm)':'mean',
                                        'Flipper Length (mm)':'mean',
                                        })
spec_summ.rename(columns={'Species':'Count'},inplace=True)
spec_summ

,Count,Body Mass (g),Bill Length (mm),Flipper Length (mm)
Species,,,,
Adelie,146,3706.164384,38.823973,190.102740
Chinstrap,68,3733.088235,48.833824,195.823529
Gentoo,119,5092.436975,47.568067,217.235294


# Species x island breakdown
Build a table showing average body mass for every species/island combination that actually occurs in the data.

In [69]:
island_summ = df.groupby(['Species','island'])['Body Mass (g)'].mean().unstack()
island_summ

island,Biscoe,Dream,Torgersen
Species,,,
Adelie,3709.659091,3701.363636,3708.510638
Chinstrap,NaN,3733.088235,NaN
Gentoo,5092.436975,NaN,NaN


In [70]:
df.pivot_table(values='Body Mass (g)',index='Species',columns='island',aggfunc='mean')

island,Biscoe,Dream,Torgersen
Species,,,
Adelie,3709.659091,3701.363636,3708.510638
Chinstrap,NaN,3733.088235,NaN
Gentoo,5092.436975,NaN,NaN


# Sex comparison
For each species, compare average body mass between male and female penguins side by side.

In [71]:
df.groupby(['Species','sex'])['Body Mass (g)'].mean().unstack()

sex,Female,Male
Species,,
Adelie,3368.835616,4043.493151
Chinstrap,3527.205882,3938.970588
Gentoo,4679.741379,5484.836066


In [72]:
df.pivot_table('Body Mass (g)','Species','sex','mean')

sex,Female,Male
Species,,
Adelie,3368.835616,4043.493151
Chinstrap,3527.205882,3938.970588
Gentoo,4679.741379,5484.836066


# Year-over-year counts
Count how many penguins of each species were recorded per year, as a wide table (years as columns).

In [73]:
over_year_count = df.groupby(['Species','year'])['sex'].count().unstack()

In [74]:
df.pivot_table(values='sex',index='Species',columns='year',aggfunc='count')

year,2007,2008,2009
Species,,,
Adelie,44,50,52
Chinstrap,26,18,24
Gentoo,33,45,41


# Top/bottom records
Find the 3 heaviest and 3 lightest penguins overall, including their species and island.

In [75]:
body_mass_sorted = df.sort_values(ascending=False,by='Body Mass (g)')
top_bot_3 = pd.concat([body_mass_sorted.head(3),body_mass_sorted.tail(3)])
top_bot_3

,Species,island,Bill Length (mm),Bill Depth (mm),Flipper Length (mm),Body Mass (g),sex,year,is_outlier,body_mass_kg,bill_ratio,size_class
169,Gentoo,Biscoe,49.2,15.2,221.0,6300.0,Male,2007,False,6.30,3.236842,large
185,Gentoo,Biscoe,59.6,17.0,230.0,6050.0,Male,2007,False,6.05,3.505882,large
229,Gentoo,Biscoe,51.1,16.3,220.0,6000.0,Male,2008,False,6.00,3.134969,large
58,Adelie,Biscoe,36.5,16.6,181.0,2850.0,Female,2008,False,2.85,2.198795,small
64,Adelie,Biscoe,36.4,17.1,184.0,2850.0,Female,2008,False,2.85,2.128655,small
314,Chinstrap,Dream,46.9,16.6,192.0,2700.0,Female,2008,False,2.70,2.825301,small


# Build the summary export
Combine your species summary and species/island breakdown into one tidy DataFrame ready for export.

In [76]:
penguins_summary = pd.concat([spec_summ, island_summ], axis=1)

# Export cleaned data
Save the cleaned, transformed DataFrame to penguins_clean.csv, and the summary table to penguins_summary.csv.

In [77]:
df.to_csv('data/penguins_cleaned.csv',index=False)
penguins_summary.to_csv('data/penguins_summary.csv',index=False)

# Sanity check
Re-load both CSVs and confirm row counts and column dtypes match what you expect before sign-off.

In [83]:
df_raw = pd.read_csv('data/penguins.csv')
df_cleaned = pd.read_csv('data/penguins_cleaned.csv')
penguins_summary = pd.read_csv('data/penguins_summary.csv')

print('Raw CSV row count:     ',len(df_raw.index))
print('Cleaned CSV row count: ',len(df_cleaned.index))

Raw CSV row count:      344
Cleaned CSV row count:  333


In [82]:
print(df_cleaned[['island', 'sex', 'Species','year']].dtypes)

island       str
sex          str
Species      str
year       int64
dtype: object


In [80]:
print(penguins_summary.info())
print(t)
penguins_summary

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Count                3 non-null      int64  
 1   Body Mass (g)        3 non-null      float64
 2   Bill Length (mm)     3 non-null      float64
 3   Flipper Length (mm)  3 non-null      float64
 4   Biscoe               2 non-null      float64
 5   Dream                2 non-null      float64
 6   Torgersen            1 non-null      float64
dtypes: float64(6), int64(1)
memory usage: 300.0 bytes
None
---------------------------------------


,Count,Body Mass (g),Bill Length (mm),Flipper Length (mm),Biscoe,Dream,Torgersen
0,146,3706.164384,38.823973,190.102740,3709.659091,3701.363636,3708.510638
1,68,3733.088235,48.833824,195.823529,NaN,3733.088235,NaN
2,119,5092.436975,47.568067,217.235294,5092.436975,NaN,NaN
